W1: Site unbalance
W2: Normal 
W3: Low yield which yield is low than 80
W4: Normal
W5: Normal
W6: Normal
W7: Normal
W8: Normal
W9: Low yield which yield is low than 80
W10: Normal
W11: Normal
W12: Normal
W13: Normal
W14: Mean Trend Up
W15: Normal
W16: Normal
W17: Normal
W18: Mean Trend Down
W19: Normal
W20: Normal
W21: Normal
W22: Normal
W23: Stdev Trend Up
W24: Normal
W25: Stdev Trend Down


In [20]:
import glob
import json
import os
import pickle
from typing import Dict, List
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
import xgboost as xgb

# Chronological target sensors aligned with SmarTest hardware subflows
SENSOR_TARGETS = [
    {"index": 1, "col": "100_Main.sensor1#CP", "name": "sensor1"},
    {"index": 2, "col": "120_Main.sensor2#DS0", "name": "sensor2"},
    {"index": 3, "col": "140_Main.sensor3#IO4", "name": "sensor3"},
    {"index": 4, "col": "160_Main.sensor4#IO1", "name": "sensor4"},
    {"index": 5, "col": "180_Main.sensor5#IO2", "name": "sensor5"},
    {"index": 6, "col": "200_Main.sensor6#IO3", "name": "sensor6"},
]

METADATA_COLS = ["PID", "Lot", "Wafer", "PF", "SBin", "HBin", "Test Time"]

WAFER_ANOMALY_MAP = {
    "W01": "Site Unbalance",
    "W03": "Low Yield (<80%)",
    "W09": "Low Yield (<80%)",
    "W14": "Mean Trend Up",
    "W18": "Mean Trend Down",
    "W23": "Stdev Trend Up",
    "W25": "Stdev Trend Down",
}


def load_and_preprocess_wafer(filepath: str) -> pd.DataFrame:
    """Loads raw CSV, skips metadata rows 1-4, and injects spatial/touchdown context."""
    df = pd.read_csv(filepath, skiprows=[1, 2, 3, 4], low_memory=False)

    numeric_cols = [c for c in df.columns if c not in ["Lot", "Wafer"]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Chronological chuck progression: Touchdown index (0..19 for 80 dies)
    if "PID" in df.columns:
        df["Touchdown_Idx"] = (df["PID"] - 1) // 4
    else:
        df["Touchdown_Idx"] = df.index // 4

    # Quiescent leakage logarithm: reflects subthreshold physics
    iddq_cols = [c for c in df.columns if "IDDQ" in c]
    for ic in iddq_cols:
        df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))

    # Touchdown Prober Card baseline: average IDDQ across the 4 active sites
    key_iddq = "80000_Main.IDDQ_flow.IDDQ_A1#IO1"
    if key_iddq in df.columns:
        df["Touchdown_Mean_IDDQ_A1"] = df.groupby("Touchdown_Idx")[
            key_iddq
        ].transform("mean")
        df["Site_Relative_IDDQ_A1"] = df[key_iddq] - df["Touchdown_Mean_IDDQ_A1"]

    return df


def extract_causal_features(
    all_raw_cols: List[str], target_col: str, df: pd.DataFrame
) -> List[str]:
    """Strict temporal anti-leakage slice: strictly features measured before target_col."""
    target_idx = all_raw_cols.index(target_col)
    candidate_raw = all_raw_cols[10:target_idx]

    features = [
        col
        for col in candidate_raw
        if col not in METADATA_COLS
        and not any(s["col"] == col for s in SENSOR_TARGETS)
    ]

    additional_features = [
        "Site",
        "X",
        "Y",
        "Touchdown_Idx",
        "Touchdown_Mean_IDDQ_A1",
        "Site_Relative_IDDQ_A1",
    ]
    log_iddqs = [c for c in df.columns if c.startswith("log_IDDQ")]

    selected = [
        c
        for c in (additional_features + log_iddqs + features)
        if c in df.columns and c not in METADATA_COLS
    ]
    return list(dict.fromkeys(selected))


def get_configured_xgboost():
    """Robust XGBoost Regressor using standard squared error (Constant Hessian = 1.0)."""
    return xgb.XGBRegressor(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.08,
        subsample=0.85,
        colsample_bytree=0.80,
        objective="reg:squarederror",  # Prevents vanishing hessian divergence
        reg_alpha=0.05,
        reg_lambda=1.00,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )


def run_leave_wafer_out_cv(data_dir: str):
    """Executes 5-Fold GroupKFold Cross-Validation grouped by Wafer ID."""
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    if not filepaths:
        raise FileNotFoundError(f"No CSV files found in directory '{data_dir}'")

    print(f"Loading {len(filepaths)} wafer files for cross-validation...")
    dfs = [load_and_preprocess_wafer(fp) for fp in filepaths]
    df_all = pd.concat(dfs, ignore_index=True)
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    wafers = df_all["Wafer"].astype(str).values
    gkf = GroupKFold(n_splits=5)

    print("\n" + "=" * 80)
    print(" 5-FOLD WAFER GROUP CROSS-VALIDATION (ZERO INTER-WAFER LEAKAGE)")
    print("=" * 80)

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = sensor["col"]

        feature_cols = extract_causal_features(raw_columns, target_col, df_all)

        # For stages 2-6, ground-truth readings from preceding sensors are valid features
        for prev in SENSOR_TARGETS:
            if (
                prev["index"] < s_idx
                and prev["col"] not in feature_cols
                and prev["col"] in df_all.columns
            ):
                feature_cols.append(prev["col"])

        X = df_all[feature_cols].copy()
        y = df_all[target_col].copy()

        fold_maes, fold_r2s = [], []
        wafer_predictions = []

        for train_idx, val_idx in gkf.split(X, y, wafers):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
            val_wafers = wafers[val_idx]

            valid_train = y_train.dropna().index
            X_train, y_train = X_train.loc[valid_train], y_train.loc[valid_train]
            valid_val = y_val.dropna().index
            X_val, y_val = X_val.loc[valid_val], y_val.loc[valid_val]
            val_wafers = val_wafers[X_val.index.isin(valid_val)]

            model = get_configured_xgboost()
            model.fit(X_train, y_train)

            preds = model.predict(X_val)
            actuals = y_val.values

            mae = mean_absolute_error(actuals, preds)
            r2 = r2_score(actuals, preds)
            fold_maes.append(mae)
            fold_r2s.append(r2)

            for w, act, prd in zip(val_wafers, actuals, preds):
                wafer_predictions.append(
                    {"Wafer": w, "Actual": act, "Pred": prd, "AbsError": abs(act - prd)}
                )

        df_preds = pd.DataFrame(wafer_predictions)
        wafer_mae_summary = (
            df_preds.groupby("Wafer")["AbsError"].mean().sort_values(ascending=False)
        )
        worst_wafer = wafer_mae_summary.index[0]
        worst_mae = wafer_mae_summary.iloc[0]

        print(f"\nTarget: {target_col} (Sensor {s_idx})")
        print(f"  • Features Used: {len(feature_cols)}")
        print(
            f"  • Cross-Wafer MAE: {np.mean(fold_maes):.4f} °C (± {np.std(fold_maes):.4f})"
        )
        print(f"  • Cross-Wafer R2:  {np.mean(fold_r2s):.4f}")
        print(
            f"  • Worst Wafer:     {worst_wafer} (MAE = {worst_mae:.4f} °C) [{WAFER_ANOMALY_MAP.get(str(worst_wafer).zfill(2), 'Normal')}]"
        )


def train_final_models(
    data_dir: str,
    train_wafers: List[str],
    test_wafers: List[str],
    output_dir: str = "./models",
):
    """Trains final production models on selected wafers and exports artifacts."""
    os.makedirs(output_dir, exist_ok=True)
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    train_dfs, test_dfs = [], []
    for fp in filepaths:
        fname = os.path.basename(fp)
        df_w = load_and_preprocess_wafer(fp)
        if any(w in fname for w in train_wafers):
            train_dfs.append(df_w)
        elif any(w in fname for w in test_wafers):
            test_dfs.append(df_w)

    df_train = pd.concat(train_dfs, ignore_index=True)
    df_test = pd.concat(test_dfs, ignore_index=True)

    feature_schemas: Dict[int, List[str]] = {}
    print("\n" + "=" * 80)
    print(
        f" FINAL TRAINING ON {len(train_dfs)} WAFERS | TESTING ON {len(test_dfs)} WAFERS"
    )
    print("=" * 80)

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = sensor["col"]

        feature_cols = extract_causal_features(raw_columns, target_col, df_train)
        for prev in SENSOR_TARGETS:
            if (
                prev["index"] < s_idx
                and prev["col"] not in feature_cols
                and prev["col"] in df_train.columns
            ):
                feature_cols.append(prev["col"])

        feature_schemas[s_idx] = feature_cols

        X_tr = df_train[feature_cols].copy()
        y_tr = df_train[target_col].copy()
        X_te = df_test[feature_cols].copy()
        y_te = df_test[target_col].copy()

        v_tr = y_tr.dropna().index
        v_te = y_te.dropna().index

        model = get_configured_xgboost()
        model.fit(X_tr.loc[v_tr], y_tr.loc[v_tr])

        preds = model.predict(X_te.loc[v_te])
        actuals = y_te.loc[v_te].values

        mae = mean_absolute_error(actuals, preds)
        r2 = r2_score(actuals, preds)

        print(
            f"Stage {s_idx} ({target_col:<22}) -> Test MAE: {mae:.4f} °C | R2: {r2:.4f}"
        )

        with open(os.path.join(output_dir, f"model_sensor{s_idx}.pkl"), "wb") as f:
            pickle.dump(model, f)

    with open(os.path.join(output_dir, "feature_schema.json"), "w") as f:
        json.dump(feature_schemas, f, indent=2)
    print(f"\nSuccessfully serialized all 6 models and schema to {output_dir}/")


if __name__ == "__main__":
    # =========================================================================
    # USER CONFIGURATION (Run directly in VS Code)
    # =========================================================================
    EXECUTION_MODE = "BOTH"  # Options: "VALIDATE", "TRAIN_FINAL", "BOTH"
    DATA_DIRECTORY = "./Data"
    OUTPUT_MODEL_DIR = "./models_XGB"

    # 5 test holdouts: representative mix of abnormal and normal wafers
    TEST_WAFERS = ["W03", "W14", "W18", "W02", "W10"]
    TRAIN_WAFERS = [
        f"W{i:02d}" for i in range(1, 26) if f"W{i:02d}" not in TEST_WAFERS
    ]
    # =========================================================================

    print(f"Starting pipeline in [{EXECUTION_MODE}] mode...")

    if EXECUTION_MODE in ("VALIDATE", "BOTH"):
        print("\n>>> Running 5-Fold Leave-Wafer-Out Cross-Validation...")
        run_leave_wafer_out_cv(DATA_DIRECTORY)

    if EXECUTION_MODE in ("TRAIN_FINAL", "BOTH"):
        print("\n>>> Training Final Models & Serializing Artifacts...")
        train_final_models(
            data_dir=DATA_DIRECTORY,
            train_wafers=TRAIN_WAFERS,
            test_wafers=TEST_WAFERS,
            output_dir=OUTPUT_MODEL_DIR,
        )

    print("\nAll tasks completed successfully.")

Starting pipeline in [BOTH] mode...

>>> Running 5-Fold Leave-Wafer-Out Cross-Validation...
Loading 25 wafer files for cross-validation...


C:\Users\morga\AppData\Local\Temp\ipykernel_8508\3499687343.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\3499687343.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\3499687343.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


 5-FOLD WAFER GROUP CROSS-VALIDATION (ZERO INTER-WAFER LEAKAGE)

Target: 100_Main.sensor1#CP (Sensor 1)
  • Features Used: 36
  • Cross-Wafer MAE: 0.0262 °C (± 0.0029)
  • Cross-Wafer R2:  0.9721
  • Worst Wafer:     1 (MAE = 0.0750 °C) [Normal]

Target: 120_Main.sensor2#DS0 (Sensor 2)
  • Features Used: 537
  • Cross-Wafer MAE: 0.0408 °C (± 0.0056)
  • Cross-Wafer R2:  0.9265
  • Worst Wafer:     1 (MAE = 0.1379 °C) [Normal]

Target: 140_Main.sensor3#IO4 (Sensor 3)
  • Features Used: 1038
  • Cross-Wafer MAE: 0.0430 °C (± 0.0138)
  • Cross-Wafer R2:  0.9687
  • Worst Wafer:     1 (MAE = 0.2411 °C) [Normal]

Target: 160_Main.sensor4#IO1 (Sensor 4)
  • Features Used: 1539
  • Cross-Wafer MAE: 0.0440 °C (± 0.0071)
  • Cross-Wafer R2:  0.9379
  • Worst Wafer:     2 (MAE = 0.1361 °C) [Normal]

Target: 180_Main.sensor5#IO2 (Sensor 5)
  • Features Used: 2040
  • Cross-Wafer MAE: 0.0435 °C (± 0.0046)
  • Cross-Wafer R2:  0.9353
  • Worst Wafer:     2 (MAE = 0.1058 °C) [Normal]

Target: 200_M

C:\Users\morga\AppData\Local\Temp\ipykernel_8508\3499687343.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\3499687343.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\3499687343.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


 FINAL TRAINING ON 20 WAFERS | TESTING ON 5 WAFERS
Stage 1 (100_Main.sensor1#CP   ) -> Test MAE: 0.0240 °C | R2: 0.9653
Stage 2 (120_Main.sensor2#DS0  ) -> Test MAE: 0.0462 °C | R2: 0.8957
Stage 3 (140_Main.sensor3#IO4  ) -> Test MAE: 0.0504 °C | R2: 0.9913
Stage 4 (160_Main.sensor4#IO1  ) -> Test MAE: 0.0584 °C | R2: 0.9487
Stage 5 (180_Main.sensor5#IO2  ) -> Test MAE: 0.0538 °C | R2: 0.9306
Stage 6 (200_Main.sensor6#IO3  ) -> Test MAE: 0.0559 °C | R2: 0.9228

Successfully serialized all 6 models and schema to ./models_XGB/

All tasks completed successfully.


In [24]:
import glob
import json
import os
import pickle
import re
from typing import Dict, List
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

# Chronological target sensors aligned with SmarTest hardware subflows
SENSOR_TARGETS = [
    {"index": 1, "col": "100_Main.sensor1#CP", "name": "sensor1"},
    {"index": 2, "col": "120_Main.sensor2#DS0", "name": "sensor2"},
    {"index": 3, "col": "140_Main.sensor3#IO4", "name": "sensor3"},
    {"index": 4, "col": "160_Main.sensor4#IO1", "name": "sensor4"},
    {"index": 5, "col": "180_Main.sensor5#IO2", "name": "sensor5"},
    {"index": 6, "col": "200_Main.sensor6#IO3", "name": "sensor6"},
]

METADATA_COLS = ["PID", "Lot", "Wafer", "PF", "SBin", "HBin", "Test Time"]

WAFER_ANOMALY_MAP = {
    "W01": "Site Unbalance",
    "W03": "Low Yield (<80%)",
    "W09": "Low Yield (<80%)",
    "W14": "Mean Trend Up",
    "W18": "Mean Trend Down",
    "W23": "Stdev Trend Up",
    "W25": "Stdev Trend Down",
}


def sanitize_column_name(col_name: str) -> str:
    """Removes special JSON characters that LightGBM's C++ core disallows."""
    return re.sub(r"[\[\]\{\}:\",]", "_", col_name)


def load_and_preprocess_wafer(filepath: str) -> pd.DataFrame:
    """Loads raw CSV, strips metadata rows 1-4, sanitizes headers, and injects features."""
    df = pd.read_csv(filepath, skiprows=[1, 2, 3, 4], low_memory=False)

    # Sanitize all column names for LightGBM compatibility
    df.columns = [sanitize_column_name(c) for c in df.columns]

    numeric_cols = [c for c in df.columns if c not in ["Lot", "Wafer"]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 1. Chronological chuck progression (0..19 touchdowns across 80 dies)
    if "PID" in df.columns:
        df["Touchdown_Idx"] = (df["PID"] - 1) // 4
    else:
        df["Touchdown_Idx"] = df.index // 4

    # 2. Quiescent leakage physics: log(IDDQ)
    iddq_cols = [c for c in df.columns if "IDDQ" in c]
    for ic in iddq_cols:
        df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))

    # 3. Touchdown Prober Baseline across the 4 active sites
    key_iddq = sanitize_column_name("80000_Main.IDDQ_flow.IDDQ_A1#IO1")
    if key_iddq in df.columns:
        df["Touchdown_Mean_IDDQ_A1"] = df.groupby("Touchdown_Idx")[
            key_iddq
        ].transform("mean")
        df["Site_Relative_IDDQ_A1"] = df[key_iddq] - df["Touchdown_Mean_IDDQ_A1"]

    return df


def extract_causal_features(
    all_raw_cols: List[str], target_col: str, df: pd.DataFrame
) -> List[str]:
    """Strict temporal anti-leakage slice: strictly features measured before target_col."""
    sanitized_raw = [sanitize_column_name(c) for c in all_raw_cols]
    sanitized_target = sanitize_column_name(target_col)
    target_idx = sanitized_raw.index(sanitized_target)

    candidate_raw = sanitized_raw[10:target_idx]
    sanitized_sensor_cols = [
        sanitize_column_name(s["col"]) for s in SENSOR_TARGETS
    ]

    features = [
        col
        for col in candidate_raw
        if col not in METADATA_COLS and col not in sanitized_sensor_cols
    ]

    additional_features = [
        "Site",
        "X",
        "Y",
        "Touchdown_Idx",
        "Touchdown_Mean_IDDQ_A1",
        "Site_Relative_IDDQ_A1",
    ]
    log_iddqs = [c for c in df.columns if c.startswith("log_IDDQ")]

    selected = [
        c
        for c in (additional_features + log_iddqs + features)
        if c in df.columns and c not in METADATA_COLS
    ]
    return list(dict.fromkeys(selected))


def get_configured_lgbm():
    """Configures LightGBM Regressor for sub-millisecond, low-variance edge inference."""
    return lgb.LGBMRegressor(
        n_estimators=60,
        max_depth=4,
        num_leaves=12,
        min_child_samples=15,
        learning_rate=0.06,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.80,
        reg_alpha=0.05,
        reg_lambda=1.20,
        objective="regression",  # Stable L2 loss
        n_jobs=-1,
        random_state=42,
        verbosity=-1,
    )


def run_leave_wafer_out_cv(data_dir: str):
    """Executes 5-Fold GroupKFold Cross-Validation grouped by Wafer ID."""
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    if not filepaths:
        raise FileNotFoundError(f"No CSV files found in directory '{data_dir}'")

    print(f"Loading {len(filepaths)} wafer files for cross-validation...")
    dfs = [load_and_preprocess_wafer(fp) for fp in filepaths]
    df_all = pd.concat(dfs, ignore_index=True)
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    wafers = df_all["Wafer"].astype(str).values
    gkf = GroupKFold(n_splits=5)

    print("\n" + "=" * 80)
    print(" 5-FOLD WAFER GROUP CROSS-VALIDATION (LIGHTGBM)")
    print("=" * 80)

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = sanitize_column_name(sensor["col"])

        feature_cols = extract_causal_features(raw_columns, target_col, df_all)

        # Allow preceding sensors causally
        for prev in SENSOR_TARGETS:
            prev_col = sanitize_column_name(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in feature_cols
                and prev_col in df_all.columns
            ):
                feature_cols.append(prev_col)

        X = df_all[feature_cols].copy()
        y = df_all[target_col].copy()

        fold_maes, fold_r2s = [], []
        wafer_predictions = []

        for train_idx, val_idx in gkf.split(X, y, wafers):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
            val_wafers = wafers[val_idx]

            valid_train = y_train.dropna().index
            X_train, y_train = X_train.loc[valid_train], y_train.loc[valid_train]
            valid_val = y_val.dropna().index
            X_val, y_val = X_val.loc[valid_val], y_val.loc[valid_val]
            val_wafers = val_wafers[X_val.index.isin(valid_val)]

            model = get_configured_lgbm()
            model.fit(X_train, y_train)

            preds = model.predict(X_val)
            actuals = y_val.values

            mae = mean_absolute_error(actuals, preds)
            r2 = r2_score(actuals, preds)
            fold_maes.append(mae)
            fold_r2s.append(r2)

            for w, act, prd in zip(val_wafers, actuals, preds):
                wafer_predictions.append(
                    {"Wafer": w, "Actual": act, "Pred": prd, "AbsError": abs(act - prd)}
                )

        df_preds = pd.DataFrame(wafer_predictions)
        wafer_mae_summary = (
            df_preds.groupby("Wafer")["AbsError"].mean().sort_values(ascending=False)
        )
        worst_wafer = wafer_mae_summary.index[0]
        worst_mae = wafer_mae_summary.iloc[0]

        print(f"\nTarget: {target_col} (Sensor {s_idx})")
        print(f"  • Features Used: {len(feature_cols)}")
        print(
            f"  • Cross-Wafer MAE: {np.mean(fold_maes):.4f} °C (± {np.std(fold_maes):.4f})"
        )
        print(f"  • Cross-Wafer R2:  {np.mean(fold_r2s):.4f}")
        print(
            f"  • Worst Wafer:     {worst_wafer} (MAE = {worst_mae:.4f} °C) [{WAFER_ANOMALY_MAP.get(str(worst_wafer).zfill(2), 'Normal')}]"
        )


def train_final_models(
    data_dir: str,
    train_wafers: List[str],
    test_wafers: List[str],
    output_dir: str = "./models",
):
    """Trains final production LightGBM models on selected wafers and exports artifacts."""
    os.makedirs(output_dir, exist_ok=True)
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    train_dfs, test_dfs = [], []
    for fp in filepaths:
        fname = os.path.basename(fp)
        df_w = load_and_preprocess_wafer(fp)
        if any(w in fname for w in train_wafers):
            train_dfs.append(df_w)
        elif any(w in fname for w in test_wafers):
            test_dfs.append(df_w)

    df_train = pd.concat(train_dfs, ignore_index=True)
    df_test = pd.concat(test_dfs, ignore_index=True)

    feature_schemas: Dict[int, List[str]] = {}
    print("\n" + "=" * 80)
    print(
        f" FINAL TRAINING ON {len(train_dfs)} WAFERS | TESTING ON {len(test_dfs)} WAFERS (LIGHTGBM)"
    )
    print("=" * 80)

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = sanitize_column_name(sensor["col"])

        feature_cols = extract_causal_features(raw_columns, target_col, df_train)
        for prev in SENSOR_TARGETS:
            prev_col = sanitize_column_name(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in feature_cols
                and prev_col in df_train.columns
            ):
                feature_cols.append(prev_col)

        feature_schemas[s_idx] = feature_cols

        X_tr = df_train[feature_cols].copy()
        y_tr = df_train[target_col].copy()
        X_te = df_test[feature_cols].copy()
        y_te = df_test[target_col].copy()

        v_tr = y_tr.dropna().index
        v_te = y_te.dropna().index

        model = get_configured_lgbm()
        model.fit(X_tr.loc[v_tr], y_tr.loc[v_tr])

        preds = model.predict(X_te.loc[v_te])
        actuals = y_te.loc[v_te].values

        mae = mean_absolute_error(actuals, preds)
        r2 = r2_score(actuals, preds)

        print(
            f"Stage {s_idx} ({target_col:<22}) -> Test MAE: {mae:.4f} °C | R2: {r2:.4f}"
        )

        with open(os.path.join(output_dir, f"model_sensor{s_idx}.pkl"), "wb") as f:
            pickle.dump(model, f)

    with open(os.path.join(output_dir, "feature_schema.json"), "w") as f:
        json.dump(feature_schemas, f, indent=2)
    print(
        f"\nSuccessfully serialized all 6 LightGBM models and schema to {output_dir}/"
    )


if __name__ == "__main__":
    # =========================================================================
    # USER CONFIGURATION (Run directly in VS Code)
    # =========================================================================
    EXECUTION_MODE = "BOTH"  # Options: "VALIDATE", "TRAIN_FINAL", "BOTH"
    DATA_DIRECTORY = "./Data"
    OUTPUT_MODEL_DIR = "./models_LightGBM"

    # 5 test holdouts: representative mix of abnormal and normal wafers
    TEST_WAFERS = ["W03", "W14", "W18", "W02", "W10"]
    TRAIN_WAFERS = [
        f"W{i:02d}" for i in range(1, 26) if f"W{i:02d}" not in TEST_WAFERS
    ]
    # =========================================================================

    print(f"Starting LightGBM pipeline in [{EXECUTION_MODE}] mode...")

    if EXECUTION_MODE in ("VALIDATE", "BOTH"):
        print("\n>>> Running 5-Fold Leave-Wafer-Out Cross-Validation...")
        run_leave_wafer_out_cv(DATA_DIRECTORY)

    if EXECUTION_MODE in ("TRAIN_FINAL", "BOTH"):
        print("\n>>> Training Final Models & Serializing Artifacts...")
        train_final_models(
            data_dir=DATA_DIRECTORY,
            train_wafers=TRAIN_WAFERS,
            test_wafers=TEST_WAFERS,
            output_dir=OUTPUT_MODEL_DIR,
        )

    print("\nAll tasks completed successfully.")

Starting LightGBM pipeline in [BOTH] mode...

>>> Running 5-Fold Leave-Wafer-Out Cross-Validation...
Loading 25 wafer files for cross-validation...


C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


 5-FOLD WAFER GROUP CROSS-VALIDATION (LIGHTGBM)

Target: 100_Main.sensor1#CP (Sensor 1)
  • Features Used: 36
  • Cross-Wafer MAE: 0.0237 °C (± 0.0030)
  • Cross-Wafer R2:  0.9751
  • Worst Wafer:     1 (MAE = 0.0725 °C) [Normal]

Target: 120_Main.sensor2#DS0 (Sensor 2)
  • Features Used: 537
  • Cross-Wafer MAE: 0.0376 °C (± 0.0056)
  • Cross-Wafer R2:  0.9339
  • Worst Wafer:     1 (MAE = 0.1331 °C) [Normal]

Target: 140_Main.sensor3#IO4 (Sensor 3)
  • Features Used: 1038
  • Cross-Wafer MAE: 0.0448 °C (± 0.0183)
  • Cross-Wafer R2:  0.9668
  • Worst Wafer:     1 (MAE = 0.2995 °C) [Normal]

Target: 160_Main.sensor4#IO1 (Sensor 4)
  • Features Used: 1539
  • Cross-Wafer MAE: 0.0432 °C (± 0.0100)
  • Cross-Wafer R2:  0.9381
  • Worst Wafer:     2 (MAE = 0.1735 °C) [Normal]

Target: 180_Main.sensor5#IO2 (Sensor 5)
  • Features Used: 2040
  • Cross-Wafer MAE: 0.0406 °C (± 0.0037)
  • Cross-Wafer R2:  0.9429
  • Worst Wafer:     2 (MAE = 0.0958 °C) [Normal]

Target: 200_Main.sensor6#IO3 

C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


 FINAL TRAINING ON 20 WAFERS | TESTING ON 5 WAFERS (LIGHTGBM)
Stage 1 (100_Main.sensor1#CP   ) -> Test MAE: 0.0249 °C | R2: 0.9574
Stage 2 (120_Main.sensor2#DS0  ) -> Test MAE: 0.0389 °C | R2: 0.9354
Stage 3 (140_Main.sensor3#IO4  ) -> Test MAE: 0.0412 °C | R2: 0.9961
Stage 4 (160_Main.sensor4#IO1  ) -> Test MAE: 0.0618 °C | R2: 0.9342
Stage 5 (180_Main.sensor5#IO2  ) -> Test MAE: 0.0483 °C | R2: 0.9422
Stage 6 (200_Main.sensor6#IO3  ) -> Test MAE: 0.0494 °C | R2: 0.9405

Successfully serialized all 6 LightGBM models and schema to ./models_LightGBM/

All tasks completed successfully.


In [28]:
import glob
import os
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

# 沿用前述前處理函式：load_and_preprocess_wafer, extract_causal_features, SENSOR_TARGETS


def tune_lightgbm_hyperparameters(
    data_dir="./", n_iter=25, target_sensor_idx=1
):
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    dfs = [load_and_preprocess_wafer(fp) for fp in filepaths]
    df_all = pd.concat(dfs, ignore_index=True)
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    sensor = next(
        s for s in SENSOR_TARGETS if s["index"] == target_sensor_idx
    )
    target_col = sanitize_column_name(sensor["col"])

    feature_cols = extract_causal_features(raw_columns, target_col, df_all)
    for prev in SENSOR_TARGETS:
        prev_col = sanitize_column_name(prev["col"])
        if (
            prev["index"] < target_sensor_idx
            and prev_col not in feature_cols
            and prev_col in df_all.columns
        ):
            feature_cols.append(prev_col)

    X = df_all[feature_cols]
    y = df_all[target_col]
    wafers = df_all["Wafer"].astype(str).values

    valid_idx = y.dropna().index
    X, y, wafers = X.loc[valid_idx], y.loc[valid_idx], wafers[valid_idx]

    # 定義搜尋網格
    param_dist = {
        "num_leaves": [7, 10, 14, 18, 24],
        "max_depth": [3, 4, 5, 6],
        "min_child_samples": [10, 15, 20, 30],
        "learning_rate": [0.03, 0.05, 0.08, 0.12],
        "n_estimators": [40, 60, 80, 120],
        "colsample_bytree": [0.65, 0.75, 0.85, 1.0],
        "subsample": [0.8, 0.9, 1.0],
        "reg_alpha": [0.0, 0.05, 0.2, 1.0],
        "reg_lambda": [0.5, 1.0, 2.0, 5.0],
        "objective": ["regression", "huber"],
    }

    base_model = lgb.LGBMRegressor(random_state=42, verbosity=-1, n_jobs=2)
    gkf = GroupKFold(n_splits=5)

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="neg_mean_absolute_error",
        cv=gkf.split(X, y, wafers),
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )

    print(
        f"\n>>> Tuning Sensor {target_sensor_idx} ({target_col}) across {len(feature_cols)} features..."
    )
    search.fit(X, y)

    best_mae = -search.best_score_
    print(f"\n[Optimum Result] Best Cross-Wafer MAE: {best_mae:.4f} °C")
    print("Best Parameters:")
    for k, v in search.best_params_.items():
        print(f"  • {k}: {v}")

    return search.best_params_


if __name__ == "__main__":
    # 可直接指定對 Stage 1 或 Stage 6 進行 30 次隨機搜尋
    tune_lightgbm_hyperparameters(
        data_dir="./Data", n_iter=30, target_sensor_idx=1
    )
    tune_lightgbm_hyperparameters(
        data_dir="./Data", n_iter=30, target_sensor_idx=2
    )
    tune_lightgbm_hyperparameters(
        data_dir="./Data", n_iter=30, target_sensor_idx=3
    )
    tune_lightgbm_hyperparameters(
        data_dir="./Data", n_iter=30, target_sensor_idx=4
    )
    tune_lightgbm_hyperparameters(
        data_dir="./Data", n_iter=30, target_sensor_idx=5
    )
    tune_lightgbm_hyperparameters(
        data_dir="./Data", n_iter=30, target_sensor_idx=6
    )

C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


>>> Tuning Sensor 1 (100_Main.sensor1#CP) across 36 features...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

[Optimum Result] Best Cross-Wafer MAE: 0.0147 °C
Best Parameters:
  • subsample: 1.0
  • reg_lambda: 2.0
  • reg_alpha: 0.05
  • objective: regression
  • num_leaves: 10
  • n_estimators: 120
  • min_child_samples: 20
  • max_depth: 5
  • learning_rate: 0.12
  • colsample_bytree: 0.75


C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


>>> Tuning Sensor 2 (120_Main.sensor2#DS0) across 537 features...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

[Optimum Result] Best Cross-Wafer MAE: 0.0258 °C
Best Parameters:
  • subsample: 1.0
  • reg_lambda: 5.0
  • reg_alpha: 0.0
  • objective: huber
  • num_leaves: 10
  • n_estimators: 120
  • min_child_samples: 15
  • max_depth: 4
  • learning_rate: 0.12
  • colsample_bytree: 0.65


C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

...


>>> Tuning Sensor 4 (160_Main.sensor4#IO1) across 1539 features...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

[Optimum Result] Best Cross-Wafer MAE: 0.0282 °C
Best Parameters:
  • subsample: 1.0
  • reg_lambda: 2.0
  • reg_alpha: 0.05
  • objective: regression
  • num_leaves: 10
  • n_estimators: 120
  • min_child_samples: 20
  • max_depth: 5
  • learning_rate: 0.12
  • colsample_bytree: 0.75


C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


>>> Tuning Sensor 5 (180_Main.sensor5#IO2) across 2040 features...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

[Optimum Result] Best Cross-Wafer MAE: 0.0263 °C
Best Parameters:
  • subsample: 1.0
  • reg_lambda: 2.0
  • reg_alpha: 0.05
  • objective: regression
  • num_leaves: 10
  • n_estimators: 120
  • min_child_samples: 20
  • max_depth: 5
  • learning_rate: 0.12
  • colsample_bytree: 0.75


C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_8508\1486917120.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


>>> Tuning Sensor 6 (200_Main.sensor6#IO3) across 2541 features...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

[Optimum Result] Best Cross-Wafer MAE: 0.0239 °C
Best Parameters:
  • subsample: 1.0
  • reg_lambda: 2.0
  • reg_alpha: 0.05
  • objective: regression
  • num_leaves: 10
  • n_estimators: 120
  • min_child_samples: 20
  • max_depth: 5
  • learning_rate: 0.12
  • colsample_bytree: 0.75


In [6]:
import glob
import json
import os
import pickle
import re
from typing import Dict, List
import lightgbm as lgb
import numpy as np
import pandas as pd

# Chronological target sensors aligned with SmarTest hardware subflows
SENSOR_TARGETS = [
    {"index": 1, "col": "100_Main.sensor1#CP", "name": "sensor1"},
    {"index": 2, "col": "120_Main.sensor2#DS0", "name": "sensor2"},
    {"index": 3, "col": "140_Main.sensor3#IO4", "name": "sensor3"},
    {"index": 4, "col": "160_Main.sensor4#IO1", "name": "sensor4"},
    {"index": 5, "col": "180_Main.sensor5#IO2", "name": "sensor5"},
    {"index": 6, "col": "200_Main.sensor6#IO3", "name": "sensor6"},
]

METADATA_COLS = ["PID", "Lot", "Wafer", "PF", "SBin", "HBin", "Test Time"]

# 填入調優後的最佳超參數矩陣
BEST_HYPERPARAMS = {
    1: {
        "num_leaves": 10,
        "max_depth": 5,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.75,
        "subsample": 1.0,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "objective": "regression",
    },
    2: {
        "num_leaves": 10,
        "max_depth": 4,
        "min_child_samples": 15,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.65,
        "subsample": 1.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "objective": "huber",
    },
    3: {
        "num_leaves": 7,
        "max_depth": 6,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 80,
        "colsample_bytree": 0.85,
        "subsample": 0.8,
        "reg_alpha": 0.2,
        "reg_lambda": 2.0,
        "objective": "huber",
    },
    4: {
        "num_leaves": 10,
        "max_depth": 5,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.75,
        "subsample": 1.0,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "objective": "regression",
    },
    5: {
        "num_leaves": 10,
        "max_depth": 5,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.75,
        "subsample": 1.0,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "objective": "regression",
    },
    6: {
        "num_leaves": 10,
        "max_depth": 5,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.75,
        "subsample": 1.0,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "objective": "regression",
    },
}


def sanitize_column_name(col_name: str) -> str:
    """替換 LightGBM 不支援的 JSON 特殊字元"""
    return re.sub(r"[\[\]\{\}:\",]", "_", col_name)


def load_and_preprocess_wafer(filepath: str) -> pd.DataFrame:
    """載入原始 CSV，排除前 4 行定義，並加入物理特徵與 Touchdown 空間特徵"""
    df = pd.read_csv(filepath, skiprows=[1, 2, 3, 4], low_memory=False)
    df.columns = [sanitize_column_name(c) for c in df.columns]

    numeric_cols = [c for c in df.columns if c not in ["Lot", "Wafer"]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 1. 承載盤加熱推進進度：Touchdown Index (0..19)
    if "PID" in df.columns:
        df["Touchdown_Idx"] = (df["PID"] - 1) // 4
    else:
        df["Touchdown_Idx"] = df.index // 4

    # 2. 靜態漏電流對數物理轉換：log(IDDQ)
    iddq_cols = [c for c in df.columns if "IDDQ" in c]
    for ic in iddq_cols:
        df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))

    # 3. 四個 Site 針尖基準中心差分
    key_iddq = sanitize_column_name("80000_Main.IDDQ_flow.IDDQ_A1#IO1")
    if key_iddq in df.columns:
        df["Touchdown_Mean_IDDQ_A1"] = df.groupby("Touchdown_Idx")[
            key_iddq
        ].transform("mean")
        df["Site_Relative_IDDQ_A1"] = df[key_iddq] - df["Touchdown_Mean_IDDQ_A1"]

    return df


def extract_causal_features(
    all_raw_cols: List[str], target_col: str, df: pd.DataFrame
) -> List[str]:
    """嚴格因果防外洩特徵切片：僅提取目標測試項之前的量測欄位"""
    sanitized_raw = [sanitize_column_name(c) for c in all_raw_cols]
    sanitized_target = sanitize_column_name(target_col)
    target_idx = sanitized_raw.index(sanitized_target)

    candidate_raw = sanitized_raw[10:target_idx]
    sanitized_sensor_cols = [
        sanitize_column_name(s["col"]) for s in SENSOR_TARGETS
    ]

    features = [
        col
        for col in candidate_raw
        if col not in METADATA_COLS and col not in sanitized_sensor_cols
    ]

    additional_features = [
        "Site",
        "X",
        "Y",
        "Touchdown_Idx",
        "Touchdown_Mean_IDDQ_A1",
        "Site_Relative_IDDQ_A1",
    ]
    log_iddqs = [c for c in df.columns if c.startswith("log_IDDQ")]

    selected = [
        c
        for c in (additional_features + log_iddqs + features)
        if c in df.columns and c not in METADATA_COLS
    ]
    return list(dict.fromkeys(selected))


def train_production_models_all25(data_dir="./", output_dir="./models"):
    """使用全量 25 片晶圓重訓最優架構模型"""
    os.makedirs(output_dir, exist_ok=True)
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    if not filepaths:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    print(
        f"\n================================================================================"
    )
    print(
        f" TRAINING FINAL OPTIMIZED MODELS ON ALL {len(filepaths)} WAFERS (2,000 SAMPLES)"
    )
    print(
        f"================================================================================"
    )

    dfs = [load_and_preprocess_wafer(fp) for fp in filepaths]
    df_full = pd.concat(dfs, ignore_index=True)
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    feature_schemas: Dict[int, List[str]] = {}

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = sanitize_column_name(sensor["col"])

        feature_cols = extract_causal_features(raw_columns, target_col, df_full)
        for prev in SENSOR_TARGETS:
            prev_col = sanitize_column_name(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in feature_cols
                and prev_col in df_full.columns
            ):
                feature_cols.append(prev_col)

        feature_schemas[s_idx] = feature_cols

        X = df_full[feature_cols].copy()
        y = df_full[target_col].copy()
        valid_idx = y.dropna().index

        # 於全量重訓迴圈內：
        if len(feature_cols) > 80:
            # 快速以輕量 LightGBM 獲取特徵重要度
            quick_model = lgb.LGBMRegressor(
                n_estimators=40, random_state=42, verbosity=-1, n_jobs=-1
            )
            v_idx = y.dropna().index
            quick_model.fit(X.loc[v_idx], y.loc[v_idx])

            importances = pd.Series(
                quick_model.feature_importances_, index=feature_cols
            )
            selected_features = (
                importances.sort_values(ascending=False).head(80).index.tolist()
            )

            X = X[selected_features]
            feature_schemas[s_idx] = selected_features
        else:
            feature_schemas[s_idx] = feature_cols

        # 載入該 Stage 調優出的專屬最佳參數
        params = BEST_HYPERPARAMS[s_idx].copy()
        params.update({"random_state": 42, "n_jobs": -1, "verbosity": -1})

        model = lgb.LGBMRegressor(**params)
        model.fit(X.loc[valid_idx], y.loc[valid_idx])

        # 保存單一模型
        model_path = os.path.join(output_dir, f"model_sensor{s_idx}.pkl")
        with open(model_path, "wb") as f:
            pickle.dump(model, f)

        print(
            f"  [✓] Sensor {s_idx} ({target_col:<22}) -> Trained on {len(valid_idx)} dies, {len(feature_cols):>4} features"
        )

    # 保存推論特徵對照表
    schema_path = os.path.join(output_dir, "feature_schema.json")
    with open(schema_path, "w") as f:
        json.dump(feature_schemas, f, indent=2)

    print(f"\n[Success] All 6 optimized models & feature schema exported to: {output_dir}/")


if __name__ == "__main__":
    # VS Code 直接執行本腳本即可一鍵重訓並匯出最終模型
    DATA_DIRECTORY = "./Data"
    OUTPUT_DIRECTORY = "./models_Final"
    train_production_models_all25(DATA_DIRECTORY, OUTPUT_DIRECTORY)


 TRAINING FINAL OPTIMIZED MODELS ON ALL 25 WAFERS (2,000 SAMPLES)


C:\Users\morga\AppData\Local\Temp\ipykernel_34780\881381742.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\881381742.py:123: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\881381742.py:123: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

  [✓] Sensor 1 (100_Main.sensor1#CP   ) -> Trained on 2000 dies,   36 features
  [✓] Sensor 2 (120_Main.sensor2#DS0  ) -> Trained on 2000 dies,  537 features
  [✓] Sensor 3 (140_Main.sensor3#IO4  ) -> Trained on 2000 dies, 1038 features
  [✓] Sensor 4 (160_Main.sensor4#IO1  ) -> Trained on 2000 dies, 1539 features
  [✓] Sensor 5 (180_Main.sensor5#IO2  ) -> Trained on 2000 dies, 2040 features
  [✓] Sensor 6 (200_Main.sensor6#IO3  ) -> Trained on 2000 dies, 2541 features

[Success] All 6 optimized models & feature schema exported to: ./models_Final/


In [4]:
import glob
import os
import re
from typing import Dict, List
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

# 目標感測器清單
SENSOR_TARGETS = [
    {"index": 1, "col": "100_Main.sensor1#CP", "name": "sensor1"},
    {"index": 2, "col": "120_Main.sensor2#DS0", "name": "sensor2"},
    {"index": 3, "col": "140_Main.sensor3#IO4", "name": "sensor3"},
    {"index": 4, "col": "160_Main.sensor4#IO1", "name": "sensor4"},
    {"index": 5, "col": "180_Main.sensor5#IO2", "name": "sensor5"},
    {"index": 6, "col": "200_Main.sensor6#IO3", "name": "sensor6"},
]

METADATA_COLS = ["PID", "Lot", "Wafer", "PF", "SBin", "HBin", "Test Time"]

# 前一步驟調優出的最優參數矩陣
BEST_HYPERPARAMS = {
    1: {
        "num_leaves": 10,
        "max_depth": 5,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.75,
        "subsample": 1.0,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "objective": "regression",
    },
    2: {
        "num_leaves": 10,
        "max_depth": 4,
        "min_child_samples": 15,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.65,
        "subsample": 1.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "objective": "huber",
    },
    3: {
        "num_leaves": 7,
        "max_depth": 6,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 80,
        "colsample_bytree": 0.85,
        "subsample": 0.8,
        "reg_alpha": 0.2,
        "reg_lambda": 2.0,
        "objective": "huber",
    },
    4: {
        "num_leaves": 10,
        "max_depth": 5,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.75,
        "subsample": 1.0,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "objective": "regression",
    },
    5: {
        "num_leaves": 10,
        "max_depth": 5,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.75,
        "subsample": 1.0,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "objective": "regression",
    },
    6: {
        "num_leaves": 10,
        "max_depth": 5,
        "min_child_samples": 20,
        "learning_rate": 0.12,
        "n_estimators": 120,
        "colsample_bytree": 0.75,
        "subsample": 1.0,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "objective": "regression",
    },
}


def sanitize_column_name(col_name: str) -> str:
    return re.sub(r"[\[\]\{\}:\",]", "_", col_name)


def load_and_preprocess(
    filepath: str,
    enable_cross_site_impute: bool = False,
    enable_anomaly_features: bool = False,
) -> pd.DataFrame:
    df = pd.read_csv(filepath, skiprows=[1, 2, 3, 4], low_memory=False)
    df.columns = [sanitize_column_name(c) for c in df.columns]

    numeric_cols = [c for c in df.columns if c not in ["Lot", "Wafer"]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Touchdown 編號 (0..19)
    if "PID" in df.columns:
        df["Touchdown_Idx"] = (df["PID"] - 1) // 4
    else:
        df["Touchdown_Idx"] = df.index // 4

    # 基礎 Log IDDQ
    iddq_cols = [c for c in df.columns if "IDDQ" in c]
    for ic in iddq_cols:
        df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))

    key_iddq = sanitize_column_name("80000_Main.IDDQ_flow.IDDQ_A1#IO1")

    # 功能 3: 跨 Site 物理中位數容錯補償 (Robust Cross-Site Imputation)
    if enable_cross_site_impute:
        # 若某顆 Die 的 IDDQ 出現負值、NaN 或 > 1000 的接觸失效，以該 Touchdown 其餘 Site 的中位數補齊
        def impute_group(g):
            med = g[key_iddq].median()
            g[key_iddq] = g[key_iddq].apply(
                lambda v: med if (pd.isna(v) or v <= 0 or v > 1e4) else v
            )
            return g

        df = df.groupby("Touchdown_Idx", group_keys=False).apply(impute_group)

    if key_iddq in df.columns:
        df["Touchdown_Mean_IDDQ_A1"] = df.groupby("Touchdown_Idx")[
            key_iddq
        ].transform("mean")
        df["Site_Relative_IDDQ_A1"] = df[key_iddq] - df["Touchdown_Mean_IDDQ_A1"]

    # 你的想法: 異常狀態感知特徵 (Anomaly-Aware Features)
    if enable_anomaly_features:
        # 特徵 1: 當前 Touchdown 前累積的故障/異常率 (反映 Low Yield 狀態)
        if "PF" in df.columns:
            # 依時序展開：計算到目前 Touchdown 為止的累計不良率
            df["Cumulative_Fail_Rate"] = (
                (df["PF"] > 0).astype(float).expanding().mean()
            )
        else:
            df["Cumulative_Fail_Rate"] = 0.0

        # 特徵 2: 承載盤漏電飄移量 (當前 Touchdown 基準 與 晶圓前 3 個 Touchdown 基準的差距)
        # 反映 Mean Trend Up (W14) 或 Down (W18)
        initial_baseline = df.loc[df["Touchdown_Idx"] < 3, key_iddq].median()
        df["Wafer_Thermal_Drift"] = (
            df["Touchdown_Mean_IDDQ_A1"] - initial_baseline
        )

        # 特徵 3: 四 Site 離散度 (反映 Site Unbalance W01)
        df["Touchdown_Site_Std"] = df.groupby("Touchdown_Idx")[
            key_iddq
        ].transform("std")

    return df


def extract_causal_features(
    all_raw_cols: List[str], target_col: str, df: pd.DataFrame
) -> List[str]:
    sanitized_raw = [sanitize_column_name(c) for c in all_raw_cols]
    sanitized_target = sanitize_column_name(target_col)
    target_idx = sanitized_raw.index(sanitized_target)

    candidate_raw = sanitized_raw[10:target_idx]
    sanitized_sensor_cols = [
        sanitize_column_name(s["col"]) for s in SENSOR_TARGETS
    ]

    features = [
        col
        for col in candidate_raw
        if col not in METADATA_COLS and col not in sanitized_sensor_cols
    ]

    context_features = [
        "Site",
        "X",
        "Y",
        "Touchdown_Idx",
        "Touchdown_Mean_IDDQ_A1",
        "Site_Relative_IDDQ_A1",
        "Cumulative_Fail_Rate",
        "Wafer_Thermal_Drift",
        "Touchdown_Site_Std",
    ]
    log_iddqs = [c for c in df.columns if c.startswith("log_IDDQ")]

    selected = [
        c
        for c in (context_features + log_iddqs + features)
        if c in df.columns and c not in METADATA_COLS
    ]
    return list(dict.fromkeys(selected))


def evaluate_pipeline(
    data_dir: str,
    top_k: int = None,
    enable_impute: bool = False,
    enable_anomaly: bool = False,
    desc: str = "",
):
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    dfs = [
        load_and_preprocess(fp, enable_impute, enable_anomaly)
        for fp in filepaths
    ]
    df_all = pd.concat(dfs, ignore_index=True)
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    wafers = df_all["Wafer"].astype(str).values
    gkf = GroupKFold(n_splits=5)

    print("\n" + "=" * 70)
    print(f" [RUNNING EVALUATION]: {desc}")
    print("=" * 70)

    stage_maes = []

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = sanitize_column_name(sensor["col"])

        feature_cols = extract_causal_features(raw_columns, target_col, df_all)
        for prev in SENSOR_TARGETS:
            prev_col = sanitize_column_name(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in feature_cols
                and prev_col in df_all.columns
            ):
                feature_cols.append(prev_col)

        X = df_all[feature_cols].copy()
        y = df_all[target_col].copy()

        # 功能 2: Top-K 特徵篩選
        if top_k is not None and len(feature_cols) > top_k:
            # 以一組快速 LightGBM 評估特徵重要度
            quick_model = lgb.LGBMRegressor(
                n_estimators=40, random_state=42, verbosity=-1, n_jobs=-1
            )
            v_idx = y.dropna().index
            quick_model.fit(X.loc[v_idx], y.loc[v_idx])
            importances = pd.Series(
                quick_model.feature_importances_, index=feature_cols
            )
            selected_features = (
                importances.sort_values(ascending=False).head(top_k).index.tolist()
            )
            X = X[selected_features]
            n_feats = len(selected_features)
        else:
            n_feats = len(feature_cols)

        fold_maes = []
        for train_idx, val_idx in gkf.split(X, y, wafers):
            X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
            X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

            v_tr = y_tr.dropna().index
            v_va = y_va.dropna().index

            params = BEST_HYPERPARAMS[s_idx].copy()
            params.update({"random_state": 42, "n_jobs": -1, "verbosity": -1})
            model = lgb.LGBMRegressor(**params)
            model.fit(X_tr.loc[v_tr], y_tr.loc[v_tr])

            preds = model.predict(X_va.loc[v_va])
            fold_maes.append(mean_absolute_error(y_va.loc[v_va], preds))

        m_mae = np.mean(fold_maes)
        stage_maes.append(m_mae)
        print(
            f"  Sensor {s_idx} ({target_col:<22}) | Feats: {n_feats:>4} | 5-Fold MAE: {m_mae:.4f} °C"
        )

    overall_mae = np.mean(stage_maes)
    print(f"  >>> OVERALL AVERAGE MAE: {overall_mae:.4f} °C")
    return overall_mae


if __name__ == "__main__":
    DATA_DIRECTORY = "./Data"

    # 1. 基準 (Baseline - 調優後的全特徵)
    mae_base = evaluate_pipeline(
        DATA_DIRECTORY, desc="Baseline (Tuned LGBM, Full Features)"
    )

    # 2. 測試 2: Top-80 特徵精簡 (Top-80 Feature Pruning)
    mae_topk = evaluate_pipeline(
        DATA_DIRECTORY,
        top_k=80,
        desc="Experiment A: Top-80 Feature Pruning (Anti-Noise & Fast Inference)",
    )

    # 3. 測試 3: 跨 Site 物理補償 (Robust Cross-Site Imputation)
    mae_impute = evaluate_pipeline(
        DATA_DIRECTORY,
        enable_impute=True,
        desc="Experiment B: Robust Cross-Site Imputation",
    )

    # 4. 測試 1 (你的想法): 異常狀態感知 (Anomaly-Aware Features)
    mae_anomaly = evaluate_pipeline(
        DATA_DIRECTORY,
        enable_anomaly=True,
        desc="Experiment C: Anomaly-Aware Features (Drift & Defect Tracking)",
    )

    # 5. 綜合疊加最優方案 (All Ideas Combined)
    mae_combo = evaluate_pipeline(
        DATA_DIRECTORY,
        top_k=80,
        enable_impute=True,
        enable_anomaly=True,
        desc="Experiment D: Combined (Top-80 + Impute + Anomaly Aware)",
    )

    print("\n" + "=" * 70)
    print(" SUMMARY OF EXPERIMENTS (5-FOLD GROUP CROSS-VALIDATION)")
    print("=" * 70)
    print(f" Baseline MAE:                  {mae_base:.4f} °C")
    print(
        f" Exp A (Top-80 Pruning):        {mae_topk:.4f} °C (Diff: {mae_topk - mae_base:+.4f} °C)"
    )
    print(
        f" Exp B (Cross-Site Impute):     {mae_impute:.4f} °C (Diff: {mae_impute - mae_base:+.4f} °C)"
    )
    print(
        f" Exp C (Anomaly-Awareness):     {mae_anomaly:.4f} °C (Diff: {mae_anomaly - mae_base:+.4f} °C)"
    )
    print(
        f" Exp D (Combined Optimum):      {mae_combo:.4f} °C (Diff: {mae_combo - mae_base:+.4f} °C)"
    )

C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w


 [RUNNING EVALUATION]: Baseline (Tuned LGBM, Full Features)
  Sensor 1 (100_Main.sensor1#CP   ) | Feats:   36 | 5-Fold MAE: 0.0147 °C
  Sensor 2 (120_Main.sensor2#DS0  ) | Feats:  537 | 5-Fold MAE: 0.0258 °C
  Sensor 3 (140_Main.sensor3#IO4  ) | Feats: 1038 | 5-Fold MAE: 0.0299 °C
  Sensor 4 (160_Main.sensor4#IO1  ) | Feats: 1539 | 5-Fold MAE: 0.0282 °C
  Sensor 5 (180_Main.sensor5#IO2  ) | Feats: 2040 | 5-Fold MAE: 0.0263 °C
  Sensor 6 (200_Main.sensor6#IO3  ) | Feats: 2541 | 5-Fold MAE: 0.0239 °C
  >>> OVERALL AVERAGE MAE: 0.0248 °C


C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w


 [RUNNING EVALUATION]: Experiment A: Top-80 Feature Pruning (Anti-Noise & Fast Inference)
  Sensor 1 (100_Main.sensor1#CP   ) | Feats:   36 | 5-Fold MAE: 0.0147 °C
  Sensor 2 (120_Main.sensor2#DS0  ) | Feats:   80 | 5-Fold MAE: 0.0243 °C
  Sensor 3 (140_Main.sensor3#IO4  ) | Feats:   80 | 5-Fold MAE: 0.0306 °C
  Sensor 4 (160_Main.sensor4#IO1  ) | Feats:   80 | 5-Fold MAE: 0.0258 °C
  Sensor 5 (180_Main.sensor5#IO2  ) | Feats:   80 | 5-Fold MAE: 0.0249 °C
  Sensor 6 (200_Main.sensor6#IO3  ) | Feats:   80 | 5-Fold MAE: 0.0209 °C
  >>> OVERALL AVERAGE MAE: 0.0235 °C


C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w


 [RUNNING EVALUATION]: Experiment B: Robust Cross-Site Imputation
  Sensor 1 (100_Main.sensor1#CP   ) | Feats:   36 | 5-Fold MAE: 0.0147 °C
  Sensor 2 (120_Main.sensor2#DS0  ) | Feats:  537 | 5-Fold MAE: 0.0258 °C
  Sensor 3 (140_Main.sensor3#IO4  ) | Feats: 1038 | 5-Fold MAE: 0.0299 °C
  Sensor 4 (160_Main.sensor4#IO1  ) | Feats: 1539 | 5-Fold MAE: 0.0282 °C
  Sensor 5 (180_Main.sensor5#IO2  ) | Feats: 2040 | 5-Fold MAE: 0.0263 °C
  Sensor 6 (200_Main.sensor6#IO3  ) | Feats: 2541 | 5-Fold MAE: 0.0239 °C
  >>> OVERALL AVERAGE MAE: 0.0248 °C


C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w


 [RUNNING EVALUATION]: Experiment C: Anomaly-Aware Features (Drift & Defect Tracking)
  Sensor 1 (100_Main.sensor1#CP   ) | Feats:   39 | 5-Fold MAE: 0.0166 °C
  Sensor 2 (120_Main.sensor2#DS0  ) | Feats:  540 | 5-Fold MAE: 0.0262 °C
  Sensor 3 (140_Main.sensor3#IO4  ) | Feats: 1041 | 5-Fold MAE: 0.0294 °C
  Sensor 4 (160_Main.sensor4#IO1  ) | Feats: 1542 | 5-Fold MAE: 0.0292 °C
  Sensor 5 (180_Main.sensor5#IO2  ) | Feats: 2043 | 5-Fold MAE: 0.0277 °C
  Sensor 6 (200_Main.sensor6#IO3  ) | Feats: 2544 | 5-Fold MAE: 0.0236 °C
  >>> OVERALL AVERAGE MAE: 0.0254 °C


C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_34780\1391695314.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w


 [RUNNING EVALUATION]: Experiment D: Combined (Top-80 + Impute + Anomaly Aware)
  Sensor 1 (100_Main.sensor1#CP   ) | Feats:   39 | 5-Fold MAE: 0.0166 °C
  Sensor 2 (120_Main.sensor2#DS0  ) | Feats:   80 | 5-Fold MAE: 0.0249 °C
  Sensor 3 (140_Main.sensor3#IO4  ) | Feats:   80 | 5-Fold MAE: 0.0308 °C
  Sensor 4 (160_Main.sensor4#IO1  ) | Feats:   80 | 5-Fold MAE: 0.0268 °C
  Sensor 5 (180_Main.sensor5#IO2  ) | Feats:   80 | 5-Fold MAE: 0.0249 °C
  Sensor 6 (200_Main.sensor6#IO3  ) | Feats:   80 | 5-Fold MAE: 0.0208 °C
  >>> OVERALL AVERAGE MAE: 0.0241 °C

 SUMMARY OF EXPERIMENTS (5-FOLD GROUP CROSS-VALIDATION)
 Baseline MAE:                  0.0248 °C
 Exp A (Top-80 Pruning):        0.0235 °C (Diff: -0.0013 °C)
 Exp B (Cross-Site Impute):     0.0248 °C (Diff: +0.0000 °C)
 Exp C (Anomaly-Awareness):     0.0254 °C (Diff: +0.0006 °C)
 Exp D (Combined Optimum):      0.0241 °C (Diff: -0.0007 °C)
